# Helmet Violation Crawler

Crawl YouTube for Vietnamese traffic videos, extract frames, and use
**Ollama LLaVA** to collect images of people **not wearing helmets**.

**Requirements:** Kaggle notebook with **GPU enabled** and **Internet enabled**.

## 1. Environment Setup (uv + Ollama)

In [ ]:
import os, sys

# Install uv
!curl -LsSf https://astral.sh/uv/install.sh | sh
os.environ["PATH"] = os.path.expanduser("~/.local/bin") + ":" + os.environ["PATH"]

In [ ]:
# Clean slate for uv init
for f in ["pyproject.toml", "uv.lock"]:
    if os.path.exists(f):
        os.remove(f)

!uv init .
!uv add yt-dlp opencv-python-headless ollama Pillow imagehash pandas matplotlib tqdm

In [ ]:
# Install Ollama
!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
import time, subprocess

# Start Ollama server in the background
ollama_proc = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)
time.sleep(3)
print(f"Ollama server started (pid={ollama_proc.pid})")

# Pull the vision model
!ollama pull llava:7b

## 2. Configuration

Edit `config.py` to tweak search queries, frame interval, model, etc.

| Setting | Default |
|---|---|
| `OLLAMA_MODEL` | `llava:7b` |
| `MAX_VIDEOS_TOTAL` | 10 |
| `FRAME_INTERVAL` | 2 s |
| `TWO_STAGE_DETECTION` | True |
| `OUTPUT_DIR` | `output/no_helmet` |

In [ ]:
import config

# Override any defaults here if needed:
# config.MAX_VIDEOS_TOTAL = 5
# config.FRAME_INTERVAL = 3
# config.OLLAMA_MODEL = "llava:13b"

print("Model           :", config.OLLAMA_MODEL)
print("Search queries  :", len(config.SEARCH_QUERIES))
print("Max videos      :", config.MAX_VIDEOS_TOTAL)
print("Frame interval  :", config.FRAME_INTERVAL, "s")
print("Output dir      :", config.OUTPUT_DIR)

## 3. Run the Pipeline

This will:
1. Search YouTube and download videos
2. Extract frames (every N seconds, deduplicated)
3. Run two-stage LLaVA analysis on each frame
4. Save frames containing no-helmet violations

In [ ]:
from crawler import run_pipeline

df = run_pipeline()

## 4. Results

In [ ]:
if df.empty:
    print("No violations detected.")
else:
    print(f"Total violation frames: {len(df)}")
    print("\nViolations per video:")
    print(df.groupby("video_id")["violation_count"].sum().sort_values(ascending=False))
    print(f"\nResults saved to: {config.RESULTS_CSV}")
    df.head(10)

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
import os

if not df.empty:
    sample = df.head(16)
    n = len(sample)
    cols = 4
    rows = (n + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(16, 4 * rows))
    axes = axes.flatten() if n > 1 else [axes]

    for i, (_, row) in enumerate(sample.iterrows()):
        img = Image.open(row["frame_path"])
        axes[i].imshow(img)
        vid = row['video_id']
        ts = row['timestamp_s']
        vc = row['violation_count']
        axes[i].set_title(f"{vid}\n{ts}s -- {vc} violation(s)", fontsize=9)
        axes[i].axis("off")

    for j in range(i + 1, len(axes)):
        axes[j].axis("off")

    plt.suptitle("No-Helmet Violations Detected", fontsize=14, y=1.01)
    plt.tight_layout()
    plt.savefig(os.path.join(config.OUTPUT_DIR, "..", "violations_grid.png"),
               dpi=150, bbox_inches="tight")
    plt.show()
else:
    print("No violation frames to display.")

## 5. (Optional) Process Local Videos

If you already have video files, skip YouTube download and analyse them directly:

In [ ]:
# Uncomment and edit to process local videos instead:
#
# local_videos = [
#     "/kaggle/input/your-dataset/video1.mp4",
#     "/kaggle/input/your-dataset/video2.mp4",
# ]
# df_local = run_pipeline(skip_download=True, video_paths=local_videos)

## 6. Cleanup

In [ ]:
# Stop Ollama server
ollama_proc.terminate()
print("Ollama server stopped.")

# Optionally remove downloaded videos to free space:
# import shutil
# shutil.rmtree(config.DOWNLOAD_DIR, ignore_errors=True)
# print("Downloads cleaned up.")